Thoughts:
- llama and command r can decently simplify sentences zero-shot.
- add HSK vocabulary context so simplification includes more HSK1-4 vocab.
- add user-specific vocab knowledge which gets added to context. This will be complex words that the user knows (don't simplify) or complex words the user doesn't know (if it is a key word to the meaning, keep it, otherwise, simplify it)
- optional end-goals/simplification level: (1) extensive reading, keep it as simple as possible (2) a little intensive so the user can learn more words

Notebook for experimenting with bedrock LLM for text simplification

In [ ]:
import boto3
from botocore.exceptions import ClientError
from langchain_aws.llms.bedrock import BedrockLLM
from langchain_aws.chat_models import ChatBedrockConverse

In [83]:
# Create an Amazon Bedrock Runtime client.
brt = boto3.client("bedrock-runtime")

# Set the model ID
arn_cr = "cohere.command-r-v1:0"
arn_ds = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.deepseek.r1-v1:0"
arn_llama = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.meta.llama3-2-3b-instruct-v1:0"
arn_llama2 = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.meta.llama3-3-70b-instruct-v1:0"

llm_llama = ChatBedrockConverse(client=brt,
                        model_id=arn_llama,
                        provider="meta",
                        temperature=0.1,
                        max_tokens=150
                        ,)

llm_llama2 = ChatBedrockConverse(client=brt,
                        model_id=arn_llama2,
                        provider="meta",
                        temperature=0.1,
                        max_tokens=150
                        ,)

llm_cr = ChatBedrockConverse(client=brt,
                        model=arn_cr,
                        temperature=0.1,
                        max_tokens=50,)

In [86]:
llm_llama2.invoke('''请简化下面的中文句子, 然后去英文翻译简化句子: [有机构指出，根据以往经验，对于中央经济工作会需要关注会议的主题词，这决定了来年宏观经济主线以及政策导向。]''')

AIMessage(content='简化后的中文句子：机构指出，中央经济工作会的主题词决定了来年的经济主线和政策导向。\n\n英文翻译：Institutions pointed out that the theme of the Central Economic Work Conference determines the main line of the economy and policy direction for the next year.', additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '484f4b3b-a617-47ad-aef6-e1febce69a8f', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Fri, 27 Jun 2025 19:32:51 GMT', 'content-type': 'application/json', 'content-length': '483', 'connection': 'keep-alive', 'x-amzn-requestid': '484f4b3b-a617-47ad-aef6-e1febce69a8f'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [526]}, 'model_name': 'arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.meta.llama3-3-70b-instruct-v1:0'}, id='run--2b3e90e7-2240-413d-9723-dd84b7b226c1-0', usage_metadata={'input_tokens': 95, 'output_tokens': 63, 'total_tokens': 158, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}})

In [88]:
sentence = "有机构指出，根据以往经验，对于中央经济工作会需要关注会议的主题词，这决定了来年宏观经济主线以及政策导向。"

In [99]:
def sentence_pipeline(sentence):
    simplified = llm_llama.invoke(f"请简化下面的中文句子: {sentence}").content
    translated = llm_llama.invoke(f"英文翻译下面的句子: {simplified}").content
    return (sentence, simplified, translated)

In [118]:
import re
paragraph = "刺激效果喜忧参半的同时，经济学家和金融机构对于中国经济前景的分析开始分化：有观点认为中国经济回升明显，今年达到5%的目标并无压力；也有两位经济学家对中国的经济困境进行深入分析，但他们的发言在网络疯传后最终被中国各个社交媒体删除。即将召开的中共中央经济工作会预计并不会给出市场关注的赤字率（目前讨论集中在4%之上还是之下），而只会给出更加模糊的定性描述。有机构指出，根据以往经验，对于中央经济工作会需要关注会议的主题词，这决定了来年宏观经济主线以及政策导向。"

sentences = [f"请简化下面的中文句子，使其更简单: {s}" for s in re.split(r'(?<=[。！？!?．.])\s*', paragraph) if s.strip()]

In [122]:
simplified = [f"英文翻译下面的句子: {response.content}" for response in llm_llama2.batch(sentences)]
# translated = llm_llama.batch(f"英文翻译下面的句子: {simplified}").content

In [123]:
translated = [response.content for response in llm_llama2.batch(simplified)]

In [124]:
sentences, simplified, translated

(['请简化下面的中文句子，使其更简单: 刺激效果喜忧参半的同时，经济学家和金融机构对于中国经济前景的分析开始分化：有观点认为中国经济回升明显，今年达到5%的目标并无压力；也有两位经济学家对中国的经济困境进行深入分析，但他们的发言在网络疯传后最终被中国各个社交媒体删除。',
  '请简化下面的中文句子，使其更简单: 即将召开的中共中央经济工作会预计并不会给出市场关注的赤字率（目前讨论集中在4%之上还是之下），而只会给出更加模糊的定性描述。',
  '请简化下面的中文句子，使其更简单: 有机构指出，根据以往经验，对于中央经济工作会需要关注会议的主题词，这决定了来年宏观经济主线以及政策导向。'],
 ['英文翻译下面的句子: 经济学家和金融机构对于中国经济前景的分析开始分化：有观点认为中国经济回升明显，今年达到5%的目标并无压力；也有两位经济学家对中国的经济困境进行深入分析，但他们的发言在网络疯传后最终被中国各个社交媒体删除。',
  '英文翻译下面的句子: 即将召开的中共中央经济工作会议可能不会明确说明下年的财政赤字率，只会给出模糊的描述。',
  '英文翻译下面的句子: 有机构指出，根据以往经验，对于中央经济工作会需要关注会议的主题词，这决定了来年宏观经济主线以及政策导向。简化为：机构指出，中央经济工作会议的主题词决定了来年的经济政策。'],
 ["Economists and financial institutions have begun to diverge in their analysis of China's economic outlook. Some argue that China's economic rebound is evident and that achieving a 5% growth target this year will be effortless. However, two economists conducted an in-depth analysis of China's economic woes, but their comments were deleted from various Chinese social media platforms after they went viral online.",
